# 03 — Student Model Training
## Multi-Hypothesis Distillation: Eng → Swahili

**Reproduces the student training phase of:**
*"Multi-Hypothesis Distillation of Multilingual Neural Translation Models for Low-Resource Languages"*

### What this notebook does
1. Loads synthetic parallel data generated by the NLLB-200 teacher (beam/top-p/top-k/MBR)
2. Trains a small scratch Transformer (Option A ≈65 M params) or a tiny debug model (Option B)
3. Optionally fine-tunes a HuggingFace Helsinki-NLP model (Option C) as a fast baseline
4. Evaluates on FLORES+ dev **and** devtest with chrF++ and BLEU (sacreBLEU)
5. Saves checkpoints and logs results to CSV

---
### Expected folder layout
```
MHD2/
├── data/
│   ├── flores/          # dev.jsonl, devtest.jsonl
│   └── synthetic/       # eng_swh_beam_M1.jsonl … eng_swh_mbr_M10.jsonl
├── notebooks/
│   ├── 03_student_train.ipynb   ← this file
│   └── models/                  # checkpoints saved here
└── results/
    └── predictions/             # beam-decoded outputs
```

---
### Compute estimates
| Run          | Sentences | Kaggle T4   | Kaggle P100 | Lightning L4/A10 |
|---|---|---|---|---|
| Debug        | 1 k       | ~3 min      | ~2 min      | ~1 min           |
| Small        | 10 k      | ~25 min     | ~15 min     | ~10 min          |
| Paper D1_BS  | ~100 k    | ~4 h        | ~2.5 h      | ~1.5 h           |
| Paper D10_BS | ~1 M      | ~40 h       | ~25 h       | ~15 h            |

**Storage:** tokenizer ≈50 MB; checkpoint ≈250 MB (65 M params fp32) / ≈130 MB (fp16); full synthetic corpus ≈400 MB

---
### ⚠️ Common failure points (read before running)
1. **Tokenizer mismatch** — always train a fresh SentencePiece BPE on YOUR data, not a generic one.
2. **Language tags** — FLORES uses `eng_Latn` / `swh_Latn`; sacreBLEU tokenizer should be `flores200`.
3. **Source/target reversal** — synthetic `.jsonl` has `src` (English) → `tgt` (Swahili). Do NOT swap.
4. **Max length truncation** — long sentences silently truncated; inspect length histogram before training.
5. **chrF++ vs chrF** — paper reports **chrF++** (word n-gram order 2). Use `chrf(word_order=2)` in sacreBLEU.
6. **dev vs devtest** — paper evaluates primarily on **devtest** (1012 sentences). Use `dev` only for HPO.
7. **Teacher-forcing vs generation** — training uses teacher forcing; validation BLEU uses beam decode. Never evaluate with teacher forcing.
8. **Option C HF fine-tune** — Helsinki opus-mt-en-sw uses its own SentencePiece; do NOT replace with your SPM.

---
## 0. Installation & Imports

In [ ]:
# Install dependencies (Kaggle / Lightning AI)
# Uncomment on first run
# !pip install --quiet torch torchvision torchaudio
# !pip install --quiet sentencepiece sacrebleu sacremoses
# !pip install --quiet transformers datasets tqdm tensorboard
# !pip install --quiet pandas numpy matplotlib seaborn

In [ ]:
import os
import sys
import json
import random
import warnings
from pathlib import Path
from typing import List, Dict, Tuple, Optional
from collections import Counter
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence
from torch.cuda.amp import autocast, GradScaler

import sentencepiece as spm
import sacrebleu
from sacrebleu.metrics import BLEU, CHRF

# HuggingFace for Option C
from transformers import (
    MarianMTModel,
    MarianTokenizer,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
)

warnings.filterwarnings('ignore')
%matplotlib inline

# Set seed for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🔹 Using device: {device}")
if torch.cuda.is_available():
    print(f"🔹 GPU: {torch.cuda.get_device_name(0)}")
    print(f"🔹 Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

---
## 1. Path Configuration
**Modify these if running on Kaggle or Lightning AI**

In [ ]:
# === PATHS — MODIFY HERE ===
ROOT = Path("..")

# Synthetic training data
SYNTHETIC_DIR = ROOT / "data" / "synthetic"

# FLORES evaluation data
FLORES_DEV = ROOT / "data" / "flores" / "dev.jsonl"
FLORES_DEVTEST = ROOT / "data" / "flores" / "devtest.jsonl"

# Output directories
MODEL_DIR = ROOT / "notebooks" / "models"
RESULTS_DIR = ROOT / "results"
PREDS_DIR = RESULTS_DIR / "predictions"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
PREDS_DIR.mkdir(parents=True, exist_ok=True)

print("✓ Paths configured:")
print(f"  Synthetic data: {SYNTHETIC_DIR}")
print(f"  FLORES dev:     {FLORES_DEV}")
print(f"  FLORES devtest: {FLORES_DEVTEST}")
print(f"  Model save:     {MODEL_DIR}")
print(f"  Predictions:    {PREDS_DIR}")

---
## 2. Hyperparameters & Configuration

In [ ]:
@dataclass
class Config:
    """All hyperparameters in one place for easy experimentation."""
    
    # === Dataset selection ===
    DATASET: str = "beam_M1"  # Options: beam_M1, beam_M10, top_p_M10, top_k_M10, dbs_M10, mbr_M10
    MAX_SAMPLES: int = 10_000  # Set to None for full dataset (100k for M10, 10k for M1)
    
    # === Tokenizer ===
    VOCAB_SIZE: int = 32_000  # BPE vocab size (paper: 32k)
    SPM_MODEL: str = "eng_swh_spm"  # Base name for SentencePiece model
    MAX_LENGTH: int = 128  # Max tokens per sentence (paper uses 100-150)
    
    # === Model architecture (Option A: Paper-style Transformer) ===
    D_MODEL: int = 512  # Hidden dimension (paper ≈ 512)
    N_HEADS: int = 8  # Attention heads
    N_LAYERS: int = 6  # Encoder/decoder layers (6-6 gives ≈65M params)
    D_FF: int = 2048  # FFN hidden size
    DROPOUT: float = 0.3  # Dropout rate (paper: 0.3)
    
    # === Training hyperparameters (from paper repo if available) ===
    BATCH_SIZE: int = 32  # Per device (gradient_accum compensates)
    GRADIENT_ACCUM: int = 2  # Effective batch = 32 × 2 = 64
    LEARNING_RATE: float = 1e-3  # Start higher for scratch training
    WARMUP_STEPS: int = 4_000  # Linear warmup
    MAX_EPOCHS: int = 50  # Early stop if no improvement
    EARLY_STOP_PATIENCE: int = 5  # Stop if no dev improvement
    LABEL_SMOOTHING: float = 0.1  # Avoid overconfidence
    WEIGHT_DECAY: float = 0.0001
    CLIP_GRAD: float = 1.0  # Gradient clipping
    
    # === Evaluation ===
    BEAM_SIZE: int = 5  # For generation
    LENGTH_PENALTY: float = 1.0  # No length penalty (neutral)
    EVAL_EVERY: int = 1_000  # Eval every N steps
    
    # === Mixed precision ===
    USE_FP16: bool = True  # Use mixed precision (faster on T4/P100/L4)
    
    # === Misc ===
    NUM_WORKERS: int = 2
    SEED: int = 42

# Initialize config
cfg = Config()

print("🔧 Configuration:")
print(f"  Dataset:       {cfg.DATASET}")
print(f"  Max samples:   {cfg.MAX_SAMPLES}")
print(f"  Model size:    d={cfg.D_MODEL}, layers={cfg.N_LAYERS}, heads={cfg.N_HEADS}")
print(f"  Batch size:    {cfg.BATCH_SIZE} × {cfg.GRADIENT_ACCUM} (effective {cfg.BATCH_SIZE * cfg.GRADIENT_ACCUM})")
print(f"  Learning rate: {cfg.LEARNING_RATE}")
print(f"  Mixed precision: {cfg.USE_FP16}")

---
## 3. Load Synthetic Training Data

In [ ]:
def load_synthetic(jsonl_path: Path, max_samples: Optional[int] = None) -> Tuple[List[str], List[str]]:
    """
    Load synthetic parallel corpus from .jsonl format.
    
    For M > 1 files, each src_id has multiple hypotheses.
    We keep ONE hypothesis per src_id (best-of strategy: use hyp_id 0)
    OR expand all N hypotheses as individual training pairs.
    
    This notebook uses EXPAND_ALL=True to match paper: all M hyps are
    used as separate training examples (multi-hypothesis training).
    """
    EXPAND_ALL = True  # Set False to use only hyp_id==0
    
    sources, targets = [], []
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            ex = json.loads(line.strip())
            if not EXPAND_ALL and ex.get("hyp_id", 0) != 0:
                continue  # skip non-primary hypotheses
            src = ex["src"].strip()
            tgt = ex["tgt"].strip()
            if src and tgt:  # skip empty
                sources.append(src)
                targets.append(tgt)
    
    if max_samples:
        # Deterministic truncation with shuffle
        paired = list(zip(sources, targets))
        random.shuffle(paired)
        paired = paired[:max_samples]
        sources, targets = zip(*paired)
        sources, targets = list(sources), list(targets)
    
    return sources, targets


# Map dataset name to file
DATASET_FILES = {
    "beam_M1":   "eng_swh_beam_M1.jsonl",
    "beam_M10":  "eng_swh_beam_M10.jsonl",
    "top_p_M10": "eng_swh_top_p_M10.jsonl",
    "top_k_M10": "eng_swh_top_k_M10.jsonl",
    "dbs_M10":   "eng_swh_dbs_M10.jsonl",
    "mbr_M10":   "eng_swh_mbr_M10.jsonl",
}

synth_file = SYNTHETIC_DIR / DATASET_FILES[cfg.DATASET]
assert synth_file.exists(), f"❌ Synthetic file not found: {synth_file}"

train_src, train_tgt = load_synthetic(synth_file, max_samples=cfg.MAX_SAMPLES)

print(f"✓ Loaded {len(train_src)} training pairs from {synth_file.name}")
print(f"  Source sample: {train_src[0][:80]}")
print(f"  Target sample: {train_tgt[0][:80]}")

# Length histogram sanity check
src_lens = [len(s.split()) for s in train_src]
tgt_lens = [len(t.split()) for t in train_tgt]
print(f"\n📊 Source word lengths — mean: {np.mean(src_lens):.1f}, p95: {np.percentile(src_lens, 95):.0f}, max: {max(src_lens)}")
print(f"📊 Target word lengths — mean: {np.mean(tgt_lens):.1f}, p95: {np.percentile(tgt_lens, 95):.0f}, max: {max(tgt_lens)}")

In [ ]:
# === Sanity Check: Inspect 5 raw pairs before any training ===
print("=" * 70)
print("PRE-TRAINING SANITY: 5 random training examples")
print("=" * 70)
for i in random.sample(range(len(train_src)), 5):
    print(f"\n[{i}] SRC: {train_src[i][:120]}")
    print(f"[{i}] TGT: {train_tgt[i][:120]}")

---
## 4. Load FLORES Evaluation Data

In [ ]:
def load_flores(jsonl_path: Path) -> Tuple[List[str], List[str]]:
    """
    Load FLORES+ dev or devtest file.
    Keys: id, flores_id, source (English), reference (Swahili)
    """
    sources, references = [], []
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            ex = json.loads(line.strip())
            sources.append(ex["source"].strip())
            references.append(ex["reference"].strip())
    return sources, references


# FLORES+ dev (997 sentences, used for validation/HPO)
flores_dev_src, flores_dev_ref = load_flores(FLORES_DEV)

# FLORES+ devtest (1012 sentences, primary benchmark)
flores_devtest_src, flores_devtest_ref = load_flores(FLORES_DEVTEST)

print(f"✓ FLORES dev:     {len(flores_dev_src)} sentences")
print(f"✓ FLORES devtest: {len(flores_devtest_src)} sentences")
print(f"\nDev sample:")
print(f"  SRC: {flores_dev_src[0][:100]}")
print(f"  REF: {flores_dev_ref[0][:100]}")

# ⚠️ SANITY: verify FLORES references are in Swahili (not reversed)
# Swahili has characteristic word patterns: 'ya', 'wa', 'na', 'katika'
swh_markers = ['ya', 'wa', 'na', 'katika', 'kwa', 'la', 'ni']
ref_words = flores_devtest_ref[0].lower().split()
marker_found = any(w in ref_words for w in swh_markers)
print(f"\n✅ Reference looks like Swahili (common words found): {marker_found}")

---
## 5. Build SentencePiece Tokenizer

Train a joint BPE tokenizer on both source (English) and target (Swahili) text.
This matches standard NMT practice; the paper uses a shared vocabulary.

> **⚠️ Tokenizer roundtrip check is mandatory before training.** If encode→decode
> changes your text significantly, your BLEU will be artificially inflated or deflated.

In [ ]:
# Path for SPM model file
SPM_PREFIX = str(MODEL_DIR / f"{cfg.SPM_MODEL}_{cfg.DATASET}")
SPM_CORPUS = str(MODEL_DIR / "spm_train_corpus.txt")


def build_spm_tokenizer(
    sources: List[str],
    targets: List[str],
    output_prefix: str,
    corpus_path: str,
    vocab_size: int = 32_000,
) -> spm.SentencePieceProcessor:
    """
    Train a SentencePiece BPE tokenizer on the combined src+tgt corpus.
    Returns a loaded processor ready for encoding.
    """
    model_file = output_prefix + ".model"
    
    if Path(model_file).exists():
        print(f"  Tokenizer already exists at {model_file}, loading...")
    else:
        print(f"  Writing corpus ({len(sources) + len(targets)} lines)...")
        with open(corpus_path, "w", encoding="utf-8") as f:
            for s in sources:
                f.write(s + "\n")
            for t in targets:
                f.write(t + "\n")
        
        print(f"  Training SentencePiece BPE (vocab_size={vocab_size})...")
        spm.SentencePieceTrainer.train(
            input=corpus_path,
            model_prefix=output_prefix,
            vocab_size=vocab_size,
            model_type="bpe",
            character_coverage=1.0,   # Full coverage (important for non-Latin scripts)
            pad_id=0,
            unk_id=1,
            bos_id=2,
            eos_id=3,
            pad_piece="<pad>",
            unk_piece="<unk>",
            bos_piece="<s>",
            eos_piece="</s>",
            shuffle_input_sentence=True,
            num_threads=4,
        )
        print(f"  ✓ Saved tokenizer: {model_file}")
    
    sp = spm.SentencePieceProcessor(model_file=model_file)
    return sp


print("Building SentencePiece tokenizer...")
sp = build_spm_tokenizer(
    sources=train_src,
    targets=train_tgt,
    output_prefix=SPM_PREFIX,
    corpus_path=SPM_CORPUS,
    vocab_size=cfg.VOCAB_SIZE,
)

PAD_ID = sp.pad_id()   # 0
UNK_ID = sp.unk_id()   # 1
BOS_ID = sp.bos_id()   # 2
EOS_ID = sp.eos_id()   # 3
VOCAB_SIZE = sp.get_piece_size()

print(f"\n✅ Tokenizer ready: vocab={VOCAB_SIZE}, PAD={PAD_ID}, BOS={BOS_ID}, EOS={EOS_ID}")


# === SANITY: Roundtrip test ===
print("\n🔄 Roundtrip sanity check:")
for s in ["Hello, world!", "Habari yako?", train_src[0][:60]]:
    ids = sp.encode(s, add_bos=False, add_eos=False)
    decoded = sp.decode(ids)
    ok = "✅" if decoded.lower().strip() == s.lower().strip() else "⚠️  MISMATCH"
    print(f"  {ok}  '{s[:40]}' → {len(ids)} tokens → '{decoded[:40]}'")

---
## 6. Dataset & DataLoader

In [ ]:
class TranslationDataset(Dataset):
    """
    PyTorch Dataset for parallel text.
    Encodes src and tgt with SentencePiece, adds BOS/EOS.
    """
    def __init__(self, sources: List[str], targets: List[str], sp: spm.SentencePieceProcessor, max_len: int = 128):
        self.sources = sources
        self.targets = targets
        self.sp = sp
        self.max_len = max_len
    
    def __len__(self):
        return len(self.sources)
    
    def __getitem__(self, idx):
        src_text = self.sources[idx]
        tgt_text = self.targets[idx]
        
        # Encode with BOS/EOS
        src_ids = [BOS_ID] + self.sp.encode(src_text, add_bos=False, add_eos=False) + [EOS_ID]
        tgt_ids = [BOS_ID] + self.sp.encode(tgt_text, add_bos=False, add_eos=False) + [EOS_ID]
        
        # Truncate if too long
        src_ids = src_ids[:self.max_len]
        tgt_ids = tgt_ids[:self.max_len]
        
        return {
            "src": torch.tensor(src_ids, dtype=torch.long),
            "tgt": torch.tensor(tgt_ids, dtype=torch.long),
        }


def collate_fn(batch: List[Dict]) -> Dict[str, torch.Tensor]:
    """
    Collate batch: pad src and tgt to max length in this batch.
    """
    src_seqs = [ex["src"] for ex in batch]
    tgt_seqs = [ex["tgt"] for ex in batch]
    
    src_padded = pad_sequence(src_seqs, batch_first=True, padding_value=PAD_ID)
    tgt_padded = pad_sequence(tgt_seqs, batch_first=True, padding_value=PAD_ID)
    
    return {
        "src": src_padded,
        "tgt": tgt_padded,
    }


# Build datasets
train_dataset = TranslationDataset(train_src, train_tgt, sp, max_len=cfg.MAX_LENGTH)

# Split off 5% for in-distribution validation (optional, for monitoring only)
val_size = int(0.05 * len(train_dataset))
train_size = len(train_dataset) - val_size
train_dataset, val_dataset = random_split(train_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(SEED))

print(f"✓ Train size: {len(train_dataset)}, Val size: {len(val_dataset)}")

# DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=cfg.NUM_WORKERS,
    pin_memory=True if device.type == "cuda" else False,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=cfg.BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=cfg.NUM_WORKERS,
    pin_memory=True if device.type == "cuda" else False,
)

print(f"✓ Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

# Test one batch
test_batch = next(iter(train_loader))
print(f"\n📦 Sample batch shapes:")
print(f"  src: {test_batch['src'].shape}  (batch, seq_len)")
print(f"  tgt: {test_batch['tgt'].shape}")
print(f"  First src sentence (token IDs): {test_batch['src'][0][:20].tolist()}")

---
## 7. Student Transformer Model

**Option A** (active by default): Paper-style scratch Transformer (≈65M params)  
**Option B**: Tiny debug model (fast testing, ≈5M params) — swap commented config below  
**Option C**: HuggingFace fine-tuning — skip to Section 10

In [ ]:
class PositionalEncoding(nn.Module):
    """
    Sinusoidal positional encoding, as in Vaswani et al. (2017).
    """
    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 512):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer("pe", pe)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, seq_len, d_model)
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


class StudentTransformer(nn.Module):
    """
    Standard encoder-decoder Transformer student model.
    
    Architecture matches the paper target:
      - Shared embedding (enc + dec)
      - Sinusoidal positional encoding
      - 6 encoder / 6 decoder layers
      - d_model=512, nhead=8, d_ff=2048 → ≈65M params
    
    For Option B (debug), use:
      d_model=128, nhead=4, num_layers=2, d_ff=512 → ≈5M params
    """
    def __init__(
        self,
        vocab_size: int,
        d_model: int = 512,
        nhead: int = 8,
        num_encoder_layers: int = 6,
        num_decoder_layers: int = 6,
        d_ff: int = 2048,
        dropout: float = 0.3,
        max_len: int = 512,
        pad_id: int = 0,
        tie_embeddings: bool = True,
    ):
        super().__init__()
        self.d_model = d_model
        self.pad_id = pad_id
        
        # Shared embedding for src and tgt
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.pos_enc = PositionalEncoding(d_model, dropout, max_len)
        
        # Standard PyTorch Transformer (enc-dec)
        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=d_ff,
            dropout=dropout,
            batch_first=True,  # (batch, seq, d_model)
        )
        
        # Output projection
        self.output_proj = nn.Linear(d_model, vocab_size, bias=False)
        
        # Tie output projection weights to embedding (reduces params, improves quality)
        if tie_embeddings:
            self.output_proj.weight = self.embedding.weight
        
        # Initialize weights
        self._init_weights()
    
    def _init_weights(self):
        """Xavier initialization for all parameters."""
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
    
    def make_padding_mask(self, ids: torch.Tensor) -> torch.Tensor:
        """Returns True where token is PAD (to be ignored in attention)."""
        return ids == self.pad_id  # (batch, seq_len)
    
    def forward(
        self,
        src: torch.Tensor,    # (batch, src_len)
        tgt: torch.Tensor,    # (batch, tgt_len)
    ) -> torch.Tensor:
        """
        Teacher-forced forward pass.
        Returns logits of shape (batch, tgt_len, vocab_size).
        """
        src_key_padding_mask = self.make_padding_mask(src)   # (batch, src_len)
        tgt_key_padding_mask = self.make_padding_mask(tgt)   # (batch, tgt_len)
        
        # Causal mask for decoder (prevent attending to future)
        tgt_len = tgt.size(1)
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt_len, device=src.device)
        
        # Embed + scale + positional encode
        scale = self.d_model ** 0.5
        src_emb = self.pos_enc(self.embedding(src) * scale)  # (batch, src_len, d_model)
        tgt_emb = self.pos_enc(self.embedding(tgt) * scale)  # (batch, tgt_len, d_model)
        
        # Transformer forward
        out = self.transformer(
            src_emb,
            tgt_emb,
            tgt_mask=tgt_mask,
            src_key_padding_mask=src_key_padding_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=src_key_padding_mask,
        )  # (batch, tgt_len, d_model)
        
        logits = self.output_proj(out)  # (batch, tgt_len, vocab_size)
        return logits


# ==============================
# BUILD MODEL
# ==============================

# Option A — Paper-style (≈65M params with tied embeddings)
model_cfg_A = dict(d_model=512, nhead=8, num_encoder_layers=6, num_decoder_layers=6, d_ff=2048, dropout=0.3)

# Option B — Debug model (≈5M params, fast iteration)
model_cfg_B = dict(d_model=128, nhead=4, num_encoder_layers=2, num_decoder_layers=2, d_ff=512, dropout=0.1)

# Choose which to use:
SELECTED_MODEL = "A"  # Change to "B" for debug / quick testing
model_cfg = model_cfg_A if SELECTED_MODEL == "A" else model_cfg_B

model = StudentTransformer(
    vocab_size=VOCAB_SIZE,
    **model_cfg,
    max_len=cfg.MAX_LENGTH + 64,
    pad_id=PAD_ID,
    tie_embeddings=True,
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"✅ Model built (Option {SELECTED_MODEL}):")
print(f"  Total parameters:     {total_params / 1e6:.2f}M")
print(f"  Trainable parameters: {trainable_params / 1e6:.2f}M")
print(f"  d_model={model_cfg['d_model']}, layers={model_cfg['num_encoder_layers']}e/{model_cfg['num_decoder_layers']}d, heads={model_cfg['nhead']}")

---
## 8. Training Utilities: Loss, Optimizer, Scheduler

In [ ]:
class LabelSmoothingCrossEntropy(nn.Module):
    """
    Cross-entropy loss with label smoothing.
    Pads are ignored. Smoothing reduces overconfidence on teacher-noisy data.
    """
    def __init__(self, vocab_size: int, pad_id: int, smoothing: float = 0.1):
        super().__init__()
        self.vocab_size = vocab_size
        self.pad_id = pad_id
        self.smoothing = smoothing
        self.confidence = 1.0 - smoothing
    
    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        """
        logits: (batch * tgt_len, vocab_size)
        targets: (batch * tgt_len,)
        """
        # Compute soft targets
        log_probs = F.log_softmax(logits, dim=-1)
        
        with torch.no_grad():
            smooth_targets = torch.full_like(log_probs, self.smoothing / (self.vocab_size - 2))
            smooth_targets.scatter_(1, targets.unsqueeze(1), self.confidence)
            smooth_targets[:, self.pad_id] = 0  # do not smooth onto PAD
        
        # Mask PAD tokens in loss
        mask = (targets != self.pad_id).float()
        loss = -(smooth_targets * log_probs).sum(dim=-1)  # (batch * tgt_len,)
        loss = (loss * mask).sum() / mask.sum().clamp(min=1)
        return loss


def get_optimizer_and_scheduler(model, cfg):
    """
    Adam optimizer with linear warmup + inverse-sqrt decay (standard NMT schedule).
    """
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=cfg.LEARNING_RATE,
        betas=(0.9, 0.98),  # From 'Attention is All You Need'
        eps=1e-9,
        weight_decay=cfg.WEIGHT_DECAY,
    )
    
    def lr_lambda(step: int) -> float:
        """Inverse-sqrt schedule with linear warmup."""
        step = max(step, 1)
        d_model = cfg.D_MODEL
        warmup = cfg.WARMUP_STEPS
        # Scaled as in the original Transformer paper
        return (d_model ** -0.5) * min(step ** -0.5, step * warmup ** -1.5)
    
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    return optimizer, scheduler


# Instantiate loss, optimizer, scaler
criterion = LabelSmoothingCrossEntropy(VOCAB_SIZE, PAD_ID, cfg.LABEL_SMOOTHING).to(device)
optimizer, scheduler = get_optimizer_and_scheduler(model, cfg)
scaler = GradScaler(enabled=cfg.USE_FP16)

print("✓ Loss: LabelSmoothingCrossEntropy")
print("✓ Optimizer: Adam (beta1=0.9, beta2=0.98, eps=1e-9)")
print("✓ Scheduler: Linear warmup + inverse-sqrt decay")
print(f"✓ Mixed precision: {cfg.USE_FP16}")

---
## 9. Beam Search Decoding & Evaluation Metrics

In [ ]:
@torch.no_grad()
def beam_search(
    model: StudentTransformer,
    src_ids: List[int],
    beam_size: int = 5,
    max_len: int = 128,
    length_penalty: float = 1.0,
) -> List[int]:
    """
    Beam search decoding. Returns the best hypothesis (list of token IDs).
    
    This is a minimal implementation; for production, consider using HF generate().
    """
    model.eval()
    
    src = torch.tensor([src_ids], dtype=torch.long, device=device)  # (1, src_len)
    
    # Encode source once
    src_key_padding_mask = model.make_padding_mask(src)
    scale = model.d_model ** 0.5
    src_emb = model.pos_enc(model.embedding(src) * scale)
    memory = model.transformer.encoder(src_emb, src_key_padding_mask=src_key_padding_mask)
    
    # Initialize beam: list of (score, sequence)
    beams = [(0.0, [BOS_ID])]  # score, sequence
    
    for _ in range(max_len):
        new_beams = []
        
        for score, seq in beams:
            if seq[-1] == EOS_ID:
                new_beams.append((score, seq))
                continue
            
            tgt = torch.tensor([seq], dtype=torch.long, device=device)  # (1, tgt_len)
            tgt_key_padding_mask = model.make_padding_mask(tgt)
            tgt_len = tgt.size(1)
            tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt_len, device=device)
            
            tgt_emb = model.pos_enc(model.embedding(tgt) * scale)
            out = model.transformer.decoder(
                tgt_emb,
                memory,
                tgt_mask=tgt_mask,
                tgt_key_padding_mask=tgt_key_padding_mask,
                memory_key_padding_mask=src_key_padding_mask,
            )  # (1, tgt_len, d_model)
            logits = model.output_proj(out[0, -1, :])  # (vocab_size,)
            log_probs = F.log_softmax(logits, dim=-1)
            
            # Top-k next tokens
            topk_lp, topk_idx = log_probs.topk(beam_size)
            for lp, idx in zip(topk_lp.tolist(), topk_idx.tolist()):
                new_score = score + lp
                new_seq = seq + [idx]
                new_beams.append((new_score, new_seq))
        
        # Keep top beam_size
        new_beams.sort(key=lambda x: x[0] / (len(x[1]) ** length_penalty), reverse=True)
        beams = new_beams[:beam_size]
        
        # Stop if all beams end in EOS
        if all(seq[-1] == EOS_ID for _, seq in beams):
            break
    
    # Return best beam
    best_score, best_seq = beams[0]
    # Remove BOS/EOS
    if best_seq[0] == BOS_ID:
        best_seq = best_seq[1:]
    if best_seq and best_seq[-1] == EOS_ID:
        best_seq = best_seq[:-1]
    return best_seq


def evaluate_flores(model, flores_src, flores_ref, beam_size=5, desc="FLORES"):
    """
    Translate FLORES sources with beam search, compute BLEU & chrF++.
    
    IMPORTANT: Paper reports chrF++ (word_order=2), not vanilla chrF.
    """
    model.eval()
    hypotheses = []
    
    for src_text in tqdm(flores_src, desc=f"Decoding {desc}", leave=False):
        src_ids = [BOS_ID] + sp.encode(src_text, add_bos=False, add_eos=False) + [EOS_ID]
        hyp_ids = beam_search(model, src_ids, beam_size=beam_size, max_len=cfg.MAX_LENGTH)
        hyp_text = sp.decode(hyp_ids)
        hypotheses.append(hyp_text)
    
    # Compute BLEU & chrF++ with sacreBLEU
    # ⚠️ Use flores200 tokenizer for fair FLORES+ comparison
    bleu = sacrebleu.corpus_bleu(hypotheses, [flores_ref], tokenize="flores200")
    chrf = sacrebleu.corpus_chrf(hypotheses, [flores_ref], word_order=2)  # chrF++ (word order 2)
    
    return {
        "bleu": bleu.score,
        "chrf++": chrf.score,
        "hypotheses": hypotheses,
    }


print("✓ Beam search decoder ready")
print("✓ Evaluation metrics: sacreBLEU (flores200 tokenizer) + chrF++ (word_order=2)")

---
## 9b. Sanity Checks (Run Before Training)

In [ ]:
# === SANITY 1: Decode 5 teacher-generated targets directly (upper bound) ===
print("=" * 70)
print("SANITY 1: Teacher-generated Swahili targets (first 5)")
print("=" * 70)
for i in range(5):
    print(f"SRC: {train_src[i][:80]}")
    print(f"TGT: {train_tgt[i][:80]}\n")

# === SANITY 2: Forward pass (model compiles OK) ===
print("=" * 70)
print("SANITY 2: Forward pass")
print("=" * 70)
model.eval()
with torch.no_grad():
    batch = next(iter(train_loader))
    src_b = batch["src"].to(device)
    tgt_b = batch["tgt"].to(device)
    # Input to decoder is tgt without last token (teacher forcing)
    tgt_in = tgt_b[:, :-1]
    tgt_out = tgt_b[:, 1:]
    logits = model(src_b, tgt_in)
    print(f"  Logits shape: {logits.shape}  (batch, tgt_len-1, vocab)")
    loss = criterion(logits.reshape(-1, VOCAB_SIZE), tgt_out.reshape(-1))
    print(f"  Initial loss (random weights): {loss.item():.4f}  (expected ~log({VOCAB_SIZE}) = {np.log(VOCAB_SIZE):.2f})")

# === SANITY 3: Overfit 100 examples (loss should drop to near 0) ===
print("\n" + "=" * 70)
print("SANITY 3: Overfit on 100 examples (10 steps)")
print("=" * 70)
model.train()
overfit_src = train_src[:100]
overfit_tgt = train_tgt[:100]
overfit_ds = TranslationDataset(overfit_src, overfit_tgt, sp, max_len=cfg.MAX_LENGTH)
overfit_dl = DataLoader(overfit_ds, batch_size=16, shuffle=True, collate_fn=collate_fn)
overfit_opt = torch.optim.Adam(model.parameters(), lr=1e-3)

for step, batch in enumerate(overfit_dl):
    if step >= 10:
        break
    src_b = batch["src"].to(device)
    tgt_b = batch["tgt"].to(device)
    tgt_in, tgt_out = tgt_b[:, :-1], tgt_b[:, 1:]
    logits = model(src_b, tgt_in)
    loss = criterion(logits.reshape(-1, VOCAB_SIZE), tgt_out.reshape(-1))
    overfit_opt.zero_grad()
    loss.backward()
    overfit_opt.step()
    print(f"  Step {step+1:2d}: loss = {loss.item():.4f}")

print("\n✅ Loss decreasing = model is learning. Re-initialize for real training below.")

# Re-initialize model, optimizer, scheduler for real training
model = StudentTransformer(
    vocab_size=VOCAB_SIZE, **model_cfg, max_len=cfg.MAX_LENGTH + 64, pad_id=PAD_ID, tie_embeddings=True
).to(device)
optimizer, scheduler = get_optimizer_and_scheduler(model, cfg)
scaler = GradScaler(enabled=cfg.USE_FP16)
print("✅ Model re-initialized for clean training run.")

---
## 10. Training & Validation Loop

In [ ]:
def train_one_epoch(model, loader, optimizer, scheduler, scaler, criterion, cfg, global_step, epoch):
    """
    One full training epoch.
    Returns (avg_loss, global_step).
    """
    model.train()
    total_loss = 0.0
    n_tokens = 0
    
    pbar = tqdm(loader, desc=f"Epoch {epoch}", leave=True)
    optimizer.zero_grad()
    
    for i, batch in enumerate(pbar):
        src_b = batch["src"].to(device, non_blocking=True)
        tgt_b = batch["tgt"].to(device, non_blocking=True)
        
        # Decoder input  = tgt[:-1]  (all tokens except last)
        # Decoder target = tgt[1:]   (all tokens except BOS)
        tgt_in  = tgt_b[:, :-1]
        tgt_out = tgt_b[:, 1:]
        
        with autocast(enabled=cfg.USE_FP16):
            logits = model(src_b, tgt_in)  # (batch, tgt_len-1, vocab)
            loss = criterion(logits.reshape(-1, VOCAB_SIZE), tgt_out.reshape(-1))
            loss = loss / cfg.GRADIENT_ACCUM  # scale for gradient accumulation
        
        scaler.scale(loss).backward()
        
        if (i + 1) % cfg.GRADIENT_ACCUM == 0:
            # Unscale before clipping
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.CLIP_GRAD)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1
        
        total_loss += loss.item() * cfg.GRADIENT_ACCUM  # undo scaling for logging
        n_tokens += (tgt_out != PAD_ID).sum().item()
        pbar.set_postfix({"loss": f"{total_loss / (i+1):.4f}", "lr": f"{scheduler.get_last_lr()[0]:.2e}"})
    
    return total_loss / len(loader), global_step


@torch.no_grad()
def validate(model, loader, criterion):
    """
    Compute validation loss (cross-entropy, teacher-forced).
    """
    model.eval()
    total_loss = 0.0
    for batch in loader:
        src_b = batch["src"].to(device, non_blocking=True)
        tgt_b = batch["tgt"].to(device, non_blocking=True)
        tgt_in, tgt_out = tgt_b[:, :-1], tgt_b[:, 1:]
        with autocast(enabled=cfg.USE_FP16):
            logits = model(src_b, tgt_in)
            loss = criterion(logits.reshape(-1, VOCAB_SIZE), tgt_out.reshape(-1))
        total_loss += loss.item()
    return total_loss / len(loader)


def save_checkpoint(model, optimizer, scheduler, epoch, step, score, path):
    """Save full training state."""
    torch.save({
        "epoch": epoch,
        "global_step": step,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "best_score": score,
        "cfg": cfg,
        "model_cfg": model_cfg,
    }, path)


print("✓ Training functions ready.")

In [ ]:
# === MAIN TRAINING LOOP ===

RUN_NAME = f"student_{cfg.DATASET}_opt{SELECTED_MODEL}"
CHECKPOINT_PATH = MODEL_DIR / f"{RUN_NAME}_best.pt"

# Training log
training_log = []
best_dev_chrf = -1.0
no_improve_count = 0
global_step = 0

print(f"🚀 Starting training: {RUN_NAME}")
print(f"   Epochs: {cfg.MAX_EPOCHS} | Batch: {cfg.BATCH_SIZE}×{cfg.GRADIENT_ACCUM} | LR: {cfg.LEARNING_RATE}")
print(f"   Checkpoint: {CHECKPOINT_PATH}\n")

for epoch in range(1, cfg.MAX_EPOCHS + 1):
    # --- Train ---
    train_loss, global_step = train_one_epoch(
        model, train_loader, optimizer, scheduler, scaler, criterion, cfg, global_step, epoch
    )
    
    # --- In-distribution validation loss ---
    val_loss = validate(model, val_loader, criterion)
    
    # --- FLORES dev evaluation (beam decode, every epoch) ---
    dev_results = evaluate_flores(
        model, flores_dev_src, flores_dev_ref, beam_size=cfg.BEAM_SIZE, desc="FLORES dev"
    )
    dev_bleu = dev_results["bleu"]
    dev_chrf = dev_results["chrf++"]
    
    # --- Log ---
    log_entry = {
        "epoch": epoch,
        "global_step": global_step,
        "train_loss": round(train_loss, 4),
        "val_loss": round(val_loss, 4),
        "flores_dev_bleu": round(dev_bleu, 4),
        "flores_dev_chrf++": round(dev_chrf, 4),
    }
    training_log.append(log_entry)
    
    print(
        f"[Epoch {epoch:3d}] train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | "
        f"dev_BLEU={dev_bleu:.2f} | dev_chrF++={dev_chrf:.2f}"
    )
    
    # --- Checkpoint (best by dev chrF++) ---
    if dev_chrf > best_dev_chrf:
        best_dev_chrf = dev_chrf
        no_improve_count = 0
        save_checkpoint(model, optimizer, scheduler, epoch, global_step, dev_chrf, CHECKPOINT_PATH)
        print(f"  ✅ New best chrF++={dev_chrf:.2f} — checkpoint saved.")
    else:
        no_improve_count += 1
        print(f"  ⏸  No improvement ({no_improve_count}/{cfg.EARLY_STOP_PATIENCE})")
    
    # --- Early stopping ---
    if no_improve_count >= cfg.EARLY_STOP_PATIENCE:
        print(f"\n⛔ Early stopping at epoch {epoch} (no improvement for {cfg.EARLY_STOP_PATIENCE} epochs)")
        break

# Save training log
log_df = pd.DataFrame(training_log)
log_path = RESULTS_DIR / f"training_log_{RUN_NAME}.csv"
log_df.to_csv(log_path, index=False)
print(f"\n📄 Training log saved: {log_path}")
print(f"🏆 Best FLORES dev chrF++: {best_dev_chrf:.2f}")

---
## 11. Final Evaluation on FLORES+ devtest

Evaluate the best checkpoint on **FLORES+ devtest** (1012 sentences, primary benchmark).

In [ ]:
# Load best checkpoint
print(f"Loading best checkpoint: {CHECKPOINT_PATH}")
ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
print(f"  Best epoch: {ckpt['epoch']} | Best dev chrF++: {ckpt['best_score']:.2f}")

# Evaluate on FLORES+ devtest
devtest_results = evaluate_flores(
    model, flores_devtest_src, flores_devtest_ref, beam_size=cfg.BEAM_SIZE, desc="FLORES devtest"
)

final_bleu = devtest_results["bleu"]
final_chrf = devtest_results["chrf++"]
final_hyps = devtest_results["hypotheses"]

print("\n" + "=" * 70)
print("📊 FINAL RESULTS (FLORES+ devtest, 1012 sentences)")
print("=" * 70)
print(f"  BLEU:     {final_bleu:.2f}")
print(f"  chrF++:   {final_chrf:.2f}")
print("=" * 70)

# Save predictions
pred_path = PREDS_DIR / f"{RUN_NAME}_devtest_beam{cfg.BEAM_SIZE}.txt"
with open(pred_path, "w", encoding="utf-8") as f:
    for hyp in final_hyps:
        f.write(hyp + "\n")
print(f"\n💾 Predictions saved: {pred_path}")

# Save scores
scores_df = pd.DataFrame([{
    "run_name": RUN_NAME,
    "dataset": cfg.DATASET,
    "model_option": SELECTED_MODEL,
    "beam_size": cfg.BEAM_SIZE,
    "eval_set": "flores_devtest",
    "sacrebleu": round(final_bleu, 4),
    "chrf_pp": round(final_chrf, 4),
    "n_sentences": len(flores_devtest_src),
}])
scores_path = RESULTS_DIR / f"student_scores_{RUN_NAME}.csv"
scores_df.to_csv(scores_path, index=False)
print(f"📄 Scores saved: {scores_path}")

In [ ]:
# === Sample translations for qualitative inspection ===
print("\n" + "=" * 70)
print("📄 Sample translations (first 5)")
print("=" * 70)
for i in range(5):
    print(f"\n[{i}] SRC: {flores_devtest_src[i][:100]}")
    print(f"    REF: {flores_devtest_ref[i][:100]}")
    print(f"    HYP: {final_hyps[i][:100]}")

---
## 12. Multi-Dataset Comparison

Train and evaluate student models on all 6 synthetic datasets to reproduce Table in the paper.

In [ ]:
# ============================================================
# MULTI-DATASET EXPERIMENT RUNNER
# Set RUN_ALL = True to train all dataset variants sequentially
# WARNING: each run takes ~4h on Kaggle T4 for 100k sentences
# ============================================================

RUN_ALL = False  # Change to True to reproduce all paper rows

EXPERIMENTS = [
    # (dataset_key, description)
    ("beam_M1",   "D1_BS   — 1 beam hypothesis per source"),
    ("beam_M10",  "D10_BS  — 10 beam hypotheses per source"),
    ("top_p_M10", "D10_top_p — 10 top-p sampled hypotheses"),
    ("top_k_M10", "D10_top_k — 10 top-k sampled hypotheses"),
    ("dbs_M10",   "D10_DBS — 10 diverse beam search hypotheses"),
    ("mbr_M10",   "D10_MBR — 10 MBR-selected hypotheses"),
]

if RUN_ALL:
    all_results = []
    for dataset_key, desc in EXPERIMENTS:
        synth_file_exp = SYNTHETIC_DIR / DATASET_FILES.get(dataset_key, "")
        if not synth_file_exp.exists():
            print(f"⚠️  Skipping {desc} — file not found: {synth_file_exp}")
            continue
        
        print(f"\n{'='*70}\n🔬 Running experiment: {desc}\n{'='*70}")
        
        # Load data
        exp_src, exp_tgt = load_synthetic(synth_file_exp, max_samples=cfg.MAX_SAMPLES)
        exp_ds = TranslationDataset(exp_src, exp_tgt, sp, max_len=cfg.MAX_LENGTH)
        val_sz = int(0.05 * len(exp_ds))
        tr_sz = len(exp_ds) - val_sz
        exp_train_ds, exp_val_ds = random_split(exp_ds, [tr_sz, val_sz], generator=torch.Generator().manual_seed(SEED))
        exp_train_dl = DataLoader(exp_train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=cfg.NUM_WORKERS)
        exp_val_dl   = DataLoader(exp_val_ds,   batch_size=cfg.BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=cfg.NUM_WORKERS)
        
        # Fresh model + optimizer
        exp_model = StudentTransformer(vocab_size=VOCAB_SIZE, **model_cfg, max_len=cfg.MAX_LENGTH + 64, pad_id=PAD_ID).to(device)
        exp_opt, exp_sched = get_optimizer_and_scheduler(exp_model, cfg)
        exp_scaler = GradScaler(enabled=cfg.USE_FP16)
        exp_ckpt = MODEL_DIR / f"student_{dataset_key}_best.pt"
        
        best_chrf = -1.0
        no_imp = 0
        g_step = 0
        
        for epoch in range(1, cfg.MAX_EPOCHS + 1):
            tr_loss, g_step = train_one_epoch(exp_model, exp_train_dl, exp_opt, exp_sched, exp_scaler, criterion, cfg, g_step, epoch)
            vl_loss = validate(exp_model, exp_val_dl, criterion)
            dev_r = evaluate_flores(exp_model, flores_dev_src, flores_dev_ref, beam_size=cfg.BEAM_SIZE, desc=f"{dataset_key} dev")
            print(f"  Epoch {epoch}: train={tr_loss:.3f} val={vl_loss:.3f} dev_BLEU={dev_r['bleu']:.2f} dev_chrF++={dev_r['chrf++']:.2f}")
            if dev_r["chrf++"] > best_chrf:
                best_chrf = dev_r["chrf++"]
                no_imp = 0
                save_checkpoint(exp_model, exp_opt, exp_sched, epoch, g_step, best_chrf, exp_ckpt)
            else:
                no_imp += 1
                if no_imp >= cfg.EARLY_STOP_PATIENCE:
                    break
        
        # Load best, eval on devtest
        ckpt_data = torch.load(exp_ckpt, map_location=device)
        exp_model.load_state_dict(ckpt_data["model_state_dict"])
        dt_r = evaluate_flores(exp_model, flores_devtest_src, flores_devtest_ref, beam_size=cfg.BEAM_SIZE, desc=f"{dataset_key} devtest")
        print(f"\n  ➡ {desc}: devtest BLEU={dt_r['bleu']:.2f} | chrF++={dt_r['chrf++']:.2f}")
        
        all_results.append({
            "dataset": dataset_key,
            "description": desc,
            "sacrebleu": round(dt_r["bleu"], 4),
            "chrf_pp": round(dt_r["chrf++"], 4),
        })
    
    # Final comparison table
    results_df = pd.DataFrame(all_results)
    print("\n" + "=" * 70)
    print("📊 MULTI-DATASET COMPARISON TABLE (FLORES+ devtest)")
    print("=" * 70)
    print(results_df.to_string(index=False))
    results_df.to_csv(RESULTS_DIR / "student_all_results.csv", index=False)
    print(f"\nSaved: {RESULTS_DIR / 'student_all_results.csv'}")

else:
    print("RUN_ALL=False — using single dataset results from Section 11.")
    print("Set RUN_ALL=True to train all 6 variants.")

---
## 13. Option C — HuggingFace Fine-tuning (Fast Baseline)

Fine-tune `Helsinki-NLP/opus-mt-en-sw` with your synthetic data.
This is a strong, fast baseline that avoids training from scratch.

> ⚠️ This uses the model's OWN tokenizer. Do NOT replace it with your SentencePiece model.

In [ ]:
# === OPTION C: HuggingFace Fine-tuning ===
# Set RUN_OPTION_C = True to use this path instead of scratch training

RUN_OPTION_C = False  # Change to True to use HF fine-tuning

if RUN_OPTION_C:
    from datasets import Dataset as HFDataset
    import evaluate
    
    # ── Load Helsinki opus-mt ──
    MODEL_HF_NAME = "Helsinki-NLP/opus-mt-en-sw"
    print(f"Loading {MODEL_HF_NAME}...")
    hf_tokenizer = MarianTokenizer.from_pretrained(MODEL_HF_NAME)
    hf_model = MarianMTModel.from_pretrained(MODEL_HF_NAME)
    print(f"  Tokenizer vocab: {hf_tokenizer.vocab_size}")
    print(f"  Model params: {sum(p.numel() for p in hf_model.parameters())/1e6:.1f}M")
    
    # ── Tokenize data ──
    MAX_LEN_HF = 128
    
    def tokenize_hf(examples):
        """Tokenize for MarianMT: uses special >>swh<< language tag."""
        src_texts = examples["src"]
        tgt_texts = examples["tgt"]
        
        # ⚠️ Marian uses >>TARGET_LANG<< prefix prepended to source for some models
        # opus-mt-en-sw does NOT require a prefix (single pair), but check if needed
        model_inputs = hf_tokenizer(
            src_texts, text_target=tgt_texts,
            max_length=MAX_LEN_HF, truncation=True, padding=False
        )
        return model_inputs
    
    # Build HF dataset
    hf_data = HFDataset.from_dict({"src": train_src, "tgt": train_tgt})
    hf_data = hf_data.map(tokenize_hf, batched=True, batch_size=512, remove_columns=["src", "tgt"])
    
    # Flores dev as HF dataset
    hf_dev = HFDataset.from_dict({"src": flores_dev_src, "tgt": flores_dev_ref})
    hf_dev = hf_dev.map(tokenize_hf, batched=True, batch_size=512, remove_columns=["src", "tgt"])
    
    print(f"✓ Tokenized train: {len(hf_data)} | dev: {len(hf_dev)}")
    
    # ── Evaluation metric ──
    sacrebleu_metric = evaluate.load("sacrebleu")
    
    def compute_metrics(eval_preds):
        preds, labels = eval_preds
        if isinstance(preds, tuple):
            preds = preds[0]
        preds = np.argmax(preds, axis=-1) if preds.ndim == 3 else preds
        decoded_preds = hf_tokenizer.batch_decode(preds, skip_special_tokens=True)
        labels = np.where(labels != -100, labels, hf_tokenizer.pad_token_id)
        decoded_labels = hf_tokenizer.batch_decode(labels, skip_special_tokens=True)
        result = sacrebleu_metric.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])
        chrf_val = sacrebleu.corpus_chrf(decoded_preds, [decoded_labels], word_order=2).score
        return {"bleu": round(result["score"], 2), "chrf++": round(chrf_val, 2)}
    
    # ── Training arguments ──
    HF_OUTPUT = str(MODEL_DIR / f"hf_finetuned_{cfg.DATASET}")
    training_args = Seq2SeqTrainingArguments(
        output_dir=HF_OUTPUT,
        num_train_epochs=cfg.MAX_EPOCHS,
        per_device_train_batch_size=cfg.BATCH_SIZE,
        per_device_eval_batch_size=16,
        gradient_accumulation_steps=cfg.GRADIENT_ACCUM,
        learning_rate=5e-5,  # Lower LR for fine-tuning
        warmup_steps=cfg.WARMUP_STEPS,
        weight_decay=cfg.WEIGHT_DECAY,
        label_smoothing_factor=cfg.LABEL_SMOOTHING,
        fp16=cfg.USE_FP16,
        evaluation_strategy="epoch",
        save_strategy="best",
        load_best_model_at_end=True,
        metric_for_best_model="chrf++",
        predict_with_generate=True,
        generation_max_length=cfg.MAX_LENGTH,
        generation_num_beams=cfg.BEAM_SIZE,
        logging_steps=200,
        report_to="none",
    )
    
    data_collator = DataCollatorForSeq2Seq(hf_tokenizer, model=hf_model, pad_to_multiple_of=8)
    
    trainer = Seq2SeqTrainer(
        model=hf_model,
        args=training_args,
        train_dataset=hf_data,
        eval_dataset=hf_dev,
        tokenizer=hf_tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
    
    print("🚀 Starting HF fine-tuning...")
    trainer.train()
    
    # Evaluate on devtest
    print("\n📊 Final eval on FLORES devtest...")
    hf_devtest = HFDataset.from_dict({"src": flores_devtest_src, "tgt": flores_devtest_ref})
    hf_devtest_tok = hf_devtest.map(tokenize_hf, batched=True, remove_columns=["src", "tgt"])
    eval_results = trainer.evaluate(hf_devtest_tok)
    print(eval_results)

else:
    print("RUN_OPTION_C=False. Set to True and run this cell to use HF fine-tuning.")

---
## 14. Results Visualization & Comparison with Teacher

In [ ]:
# ── Load teacher scores for comparison ──
teacher_scores_path = RESULTS_DIR / "teacher_flores_scores.csv"
if teacher_scores_path.exists():
    teacher_df = pd.read_csv(teacher_scores_path)
    print("Teacher baseline (NLLB-200-distilled-600M on FLORES devtest):")
    print(teacher_df[["method", "M", "sacrebleu", "chrf_pp"]].to_string(index=False))
else:
    print(f"Teacher scores not found at {teacher_scores_path}")

# ── Load student scores ──
student_scores_path = RESULTS_DIR / f"student_scores_{RUN_NAME}.csv"
if student_scores_path.exists():
    student_df = pd.read_csv(student_scores_path)
    print("\nStudent results:")
    print(student_df.to_string(index=False))
    
    # Compute gap
    teacher_beam1 = teacher_df[teacher_df["method"] == "beam"].iloc[0]
    bleu_gap  = teacher_beam1["sacrebleu"] - student_df.iloc[0]["sacrebleu"]
    chrf_gap  = teacher_beam1["chrf_pp"]   - student_df.iloc[0]["chrf_pp"]
    print(f"\n📉 Student vs Teacher (D1_BS beam) gap:")
    print(f"   BLEU gap:   {bleu_gap:.2f}")
    print(f"   chrF++ gap: {chrf_gap:.2f}")

In [ ]:
# ── Training curve ──
if training_log:
    log_df = pd.DataFrame(training_log)
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    fig.suptitle(f"Training Curves — {RUN_NAME}", fontsize=13)
    
    axes[0].plot(log_df["epoch"], log_df["train_loss"], label="train", marker="o", ms=3)
    axes[0].plot(log_df["epoch"], log_df["val_loss"], label="val", marker="s", ms=3)
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].set_title("Training & Validation Loss")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    axes[1].plot(log_df["epoch"], log_df["flores_dev_bleu"], color="steelblue", marker="o", ms=3)
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("BLEU")
    axes[1].set_title("FLORES dev BLEU")
    axes[1].grid(True, alpha=0.3)
    
    axes[2].plot(log_df["epoch"], log_df["flores_dev_chrf++"], color="darkorange", marker="o", ms=3)
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel("chrF++")
    axes[2].set_title("FLORES dev chrF++")
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plot_path = RESULTS_DIR / "plots" / f"training_curves_{RUN_NAME}.png"
    plt.savefig(plot_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Plot saved: {plot_path}")
else:
    print("No training log found. Run training loop first.")

---
## 15. Diagnostics & Final Sanity Checks

In [ ]:
# === Check tokenizer vocabulary distribution ===
print("=" * 70)
print("DIAGNOSTIC: Vocabulary coverage on FLORES devtest")
print("=" * 70)

# Tokenize FLORES devtest src and tgt
all_ids_src = []
all_ids_tgt = []
for src, tgt in zip(flores_devtest_src, flores_devtest_ref):
    all_ids_src.extend(sp.encode(src, add_bos=False, add_eos=False))
    all_ids_tgt.extend(sp.encode(tgt, add_bos=False, add_eos=False))

unk_count_src = all_ids_src.count(UNK_ID)
unk_count_tgt = all_ids_tgt.count(UNK_ID)
total_src = len(all_ids_src)
total_tgt = len(all_ids_tgt)

print(f"Source UNK rate: {100 * unk_count_src / total_src:.2f}% ({unk_count_src}/{total_src})")
print(f"Target UNK rate: {100 * unk_count_tgt / total_tgt:.2f}% ({unk_count_tgt}/{total_tgt})")
print("(Expect < 1% for good coverage)\n")

# === Token length distribution vs max_length ===
print("=" * 70)
print("DIAGNOSTIC: Token length truncation")
print("=" * 70)
src_token_lens = [len(sp.encode(s, add_bos=True, add_eos=True)) for s in flores_devtest_src]
tgt_token_lens = [len(sp.encode(t, add_bos=True, add_eos=True)) for t in flores_devtest_ref]
src_truncated = sum(1 for ln in src_token_lens if ln > cfg.MAX_LENGTH)
tgt_truncated = sum(1 for ln in tgt_token_lens if ln > cfg.MAX_LENGTH)

print(f"Source: {src_truncated}/{len(flores_devtest_src)} sentences exceed max_length={cfg.MAX_LENGTH}")
print(f"Target: {tgt_truncated}/{len(flores_devtest_ref)} sentences exceed max_length={cfg.MAX_LENGTH}")
print(f"p95 src token length: {np.percentile(src_token_lens, 95):.0f}")
print(f"p95 tgt token length: {np.percentile(tgt_token_lens, 95):.0f}")

if src_truncated > 50 or tgt_truncated > 50:
    print("\n⚠️  Many sentences truncated. Consider increasing MAX_LENGTH.")
else:
    print("\n✅ Truncation is minimal.")

In [ ]:
# === Teacher-generated quality: reference vs hypothesis comparison ===
# If you have access to the teacher-generated synthetic targets, check their BLEU vs FLORES refs
print("\n" + "=" * 70)
print("DIAGNOSTIC: Teacher-generated target quality (sanity)")
print("=" * 70)
print("Compare teacher synthetic outputs to human FLORES references:")
print("If teacher-generated targets are garbage, student can't learn well.\n")

# Sample 100 training pairs
sample_idx = random.sample(range(len(train_src)), min(100, len(train_src)))
sample_tgt_synth = [train_tgt[i] for i in sample_idx]

# Try to evaluate against FLORES refs if sources overlap (unlikely, but for illustration)
# In practice, synthetic and FLORES are disjoint; this is just for internal consistency.
print("(Synthetic train tgt vs synthetic train tgt self-BLEU):")
sample_tgt_copy = sample_tgt_synth[:]  # dummy check
self_bleu = sacrebleu.corpus_bleu(sample_tgt_synth, [sample_tgt_copy], tokenize="flores200").score
print(f"  Self-BLEU (should be 100): {self_bleu:.2f}")
print("\n  For real quality check, manually inspect sample pairs or use external reference if available.")

In [ ]:
# === Verify FLORES language codes ===
print("\n" + "=" * 70)
print("DIAGNOSTIC: FLORES language code sanity")
print("=" * 70)
print("Checking that FLORES references are in Swahili (swh_Latn)...\n")
swh_markers = ['na', 'wa', 'ya', 'la', 'kwa', 'katika', 'ni', 'au', 'kama', 'haki', 'sasa']
marker_hits = 0
for ref in flores_devtest_ref[:50]:
    ref_lower = ref.lower()
    if any(w in ref_lower.split() for w in swh_markers):
        marker_hits += 1

print(f"Found Swahili markers in {marker_hits}/50 samples.")
if marker_hits < 25:
    print("⚠️  Low marker hit rate — verify FLORES references are Swahili, not English!")
else:
    print("✅ References look like Swahili.")

In [ ]:
print("\n" + "=" * 70)
print("✅ ALL DIAGNOSTICS COMPLETE")
print("=" * 70)
print("If any warnings appear above, address them before reporting final results.")
print("Otherwise, you are ready to compare student scores with the paper baseline.")

---
## 16. Summary & Next Steps

### ✅ What you have now:
1. **Trained student Transformer** (≈65M params, Option A) on synthetic English→Swahili data
2. **Evaluation on FLORES+ devtest** (BLEU & chrF++)
3. **Saved checkpoints** for reproducibility
4. **Diagnostic checks** for tokenizer, truncation, and language sanity
5. **Training logs & plots** for analysis

---
### 📊 Expected results (paper baseline):
- **Teacher (NLLB-200-600M) on FLORES devtest:**
  - BLEU ≈ 33.9 | chrF++ ≈ 60.0 (beam search M=1)
- **Student (65M scratch) on D1_BS:**
  - BLEU ≈ 25–30 | chrF++ ≈ 50–55 (expected gap ≈5–10 points)
- **Student on D10_BS (multi-hypothesis):**
  - BLEU slightly higher than D1_BS if diversity helps

Your mileage may vary based on hyperparameters and compute budget.

---
### 🚀 Next steps:
1. **Run multi-dataset comparison** (Section 12) to reproduce Table in paper
2. **Ablation studies**: test different vocab sizes, model sizes, learning rates
3. **Extend to other language pairs** (e.g., eng→zul, eng→xho)
4. **Deploy student model** for inference with `transformers` or ONNX
5. **Compare with HF Option C** (fine-tuned Helsinki opus-mt) as a strong baseline

---
### 📚 References:
- Paper: *Multi-Hypothesis Distillation of Multilingual Neural Translation Models for Low-Resource Languages* ([arXiv:2507.21568](https://arxiv.org/abs/2507.21568))
- Code: [transducens/sampling-distillation](https://github.com/transducens/sampling-distillation) (if available)
- FLORES+: [facebook/flores](https://github.com/facebookresearch/flores)
- sacreBLEU: [mjpost/sacreBLEU](https://github.com/mjpost/sacreBLEU)
- SentencePiece: [google/sentencepiece](https://github.com/google/sentencepiece)

---
**Good luck with your reproduction! 🎓**